# 🫀 ST-MEM 1D Pretraining — ChagaSight

## Run order (every session)
| Cell | Purpose | Run when |
|------|---------|----------|
| 1 | Mount Drive + GPU check | Every session |
| 2 | Configuration | Every session |
| 3 | **Quick test** — folders + CSV + direct file access | Every session |
| 4 | **Deep validator** — shapes, NaN, Inf | First time only |
| 5 | Model + Dataset + CheckpointManager | Every session |
| 6 | Training function | Every session |
| 7 | START / RESUME | Every session |
| 8 | Verify output | After training |

## Why CSV-driven?
Google Drive FUSE cannot list 342k files — `find`, `ls`, `glob` all return 0 or time out on `code15/`.  
This notebook **never lists that folder**. It reads IDs from the metadata CSV and constructs exact file paths.  
Drive can open a file given its exact path even when it cannot list the folder containing 342k files.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 1 — Mount Drive & GPU check
# ══════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

import subprocess, torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU     : {props.name}')
    print(f'VRAM    : {props.total_memory / 1e9:.1f} GB')
    torch.backends.cudnn.benchmark = True
    print('cudnn.benchmark = True  (5-10% speedup)')
else:
    print('No GPU — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 2 — Configuration
# ══════════════════════════════════════════════════════════════════

import os

DRIVE_ROOT     = '/content/drive/MyDrive/ChagaSight'
DATA_DIR       = f'{DRIVE_ROOT}/data/processed/1d_signals.100hz'
METADATA_PATH  = f'{DRIVE_ROOT}/data/processed/metadata/all_data.csv'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

CONFIG = {
    'data_dir'                 : DATA_DIR,
    'metadata_path'            : METADATA_PATH,
    'checkpoint_dir'           : CHECKPOINT_DIR,
    'subset'                   : 1.0,
    'epochs'                   : 20,
    'batch_size'               : 64,        # T4=64  A100=128
    'num_workers'              : 2,
    'lr'                       : 1.5e-4,
    'warmup_epochs'            : 2,
    'mask_ratio'               : 0.75,
    'save_every'               : 5,
    'checkpoint_every_batches' : 300,
    'use_amp'                  : True,
    'code15_local_dir'         : '/content/code15',  # staged from Drive
}

print('CONFIG:')
for k, v in CONFIG.items():
    print(f'  {k:<30} : {v}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 2b — Stage code15 to Colab local disk
#
# WHY: Drive FUSE Errno 5 — the flat code15/ folder (342k files)
#      fails BOTH directory traversal (rsync/cp/find) AND random
#      file opens. The only reliable method is to copy file-by-file
#      using exact paths constructed from the CSV (no opendir call).
#
# HOW: Reads IDs from all_data.csv, copies each {id}.npy by path.
#      ThreadPoolExecutor(8) runs 8 parallel copies -> ~3-6 min.
#      Idempotent: skips files already in /content/code15/.
#
# After Colab disconnect: re-run this cell. Skips already-copied.
# ══════════════════════════════════════════════════════════════════

import pandas as pd, shutil, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

DRIVE_CODE15 = Path(CONFIG['data_dir']) / 'code15'
LOCAL_CODE15 = Path(CONFIG['code15_local_dir'])
META_PATH    = Path(CONFIG['metadata_path'])
WORKERS      = 8   # parallel copy threads

LOCAL_CODE15.mkdir(parents=True, exist_ok=True)

print('=' * 65)
print('STAGING code15 -> /content/code15/')
print('=' * 65)
print(f'  Drive src : {DRIVE_CODE15}')
print(f'  Local dst : {LOCAL_CODE15}')
print(f'  Workers   : {WORKERS}')
print()

# Read IDs from CSV (no Drive directory traversal needed)
df = pd.read_csv(META_PATH, dtype={'id': str}, low_memory=False)
ids = df[df['dataset'] == 'code15']['id'].values
print(f'  CSV IDs   : {len(ids):,}')

# Check what's already staged
already_staged = {p.stem for p in LOCAL_CODE15.glob('*.npy')}
to_copy = [fid for fid in ids if fid not in already_staged]
print(f'  Already   : {len(already_staged):,}')
print(f'  To copy   : {len(to_copy):,}')
print()

if not to_copy:
    print('All files already staged!')
else:
    def copy_one(fid):
        src = DRIVE_CODE15 / f'{fid}.npy'
        dst = LOCAL_CODE15 / f'{fid}.npy'
        try:
            shutil.copy2(str(src), str(dst))
            return 'ok'
        except Exception as e:
            return f'fail:{e}'

    t0      = time.time()
    ok      = fail = 0
    errors  = []

    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futures = {ex.submit(copy_one, fid): fid for fid in to_copy}
        pbar    = tqdm(as_completed(futures), total=len(to_copy),
                       desc='Copying', unit='file')
        for fut in pbar:
            res = fut.result()
            if res == 'ok':
                ok += 1
            else:
                fail += 1
                errors.append(f'{futures[fut]}: {res}')
            if ok % 5000 == 0 and ok > 0:
                elapsed = time.time() - t0
                rate    = ok / elapsed
                eta_m   = (len(to_copy) - ok) / rate / 60
                pbar.set_postfix({'ok': ok, 'fail': fail,
                                  'rate': f'{rate:.0f}/s',
                                  'ETA': f'{eta_m:.1f}m'})

    elapsed = time.time() - t0
    print(f'\n  Copied  : {ok:,}  Failed: {fail}  Time: {elapsed/60:.1f} min')
    if errors[:5]:
        print(f'  Sample errors:')
        for e in errors[:5]:
            print(f'    {e}')

# Final verification
staged = len(list(LOCAL_CODE15.glob('*.npy')))
print(f'  Staged total: {staged:,} / {len(ids):,}')

if staged > 0:
    import numpy as np
    sample = next(LOCAL_CODE15.glob('*.npy'))
    arr    = np.load(str(sample))
    print(f'  Spot-check  : {sample.name} shape={arr.shape} dtype={arr.dtype}  OK')
    if staged < len(ids) * 0.95:
        print(f'  WARNING: only {staged/len(ids)*100:.1f}% staged')
        print(f'  Re-run this cell to continue from where it stopped')
    else:
        print(f'\nOK  code15 staged to local disk — training will use /content/code15/')
else:
    print('\nERROR  0 files staged.')
    print('  Possible causes:')
    print('  1. Drive mount issue — re-run Cell 1 (force_remount=True)')
    print('  2. code15/ path wrong — check CONFIG[data_dir]')
    print('  3. Drive API quota — wait 1 min and retry')

print('=' * 65)


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 3 — Quick test  (run every session before training)
#
# 1. Folder existence
# 2. CSV loads with expected columns and row counts
# 3. Load 3 files per dataset by CSV ID  (proves Drive access)
# 4. Checkpoint status
#
# Does NOT list code15/ at any point.
# ══════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import torch
from pathlib import Path

DATA_PATH = Path(CONFIG['data_dir'])
META_PATH = Path(CONFIG['metadata_path'])
CKPT_PATH = Path(CONFIG['checkpoint_dir'])
DATASETS  = ['ptbxl', 'samitrop', 'code15']

print('=' * 65)
print('QUICK TEST')
print('=' * 65)
all_ok = True

# ── 1. Folder existence ──────────────────────────────────────────
print('\n1. Dataset folders:')
for ds in DATASETS:
    p = DATA_PATH / ds
    if p.exists():
        print(f'   OK  {ds:<12} -> {p}')
    else:
        print(f'   MISSING  {ds:<12} -> {p}')
        all_ok = False

# ── 2. CSV ───────────────────────────────────────────────────────
print('\n2. Metadata CSV:')
df_meta = None
if META_PATH.exists():
    df_meta = pd.read_csv(META_PATH, dtype={'id': str}, low_memory=False)
    print(f'   OK  {META_PATH.name}')
    print(f'   Rows    : {len(df_meta):,}')
    print(f'   Columns : {df_meta.columns.tolist()}')
    if 'dataset' in df_meta.columns:
        for ds in DATASETS:
            n = len(df_meta[df_meta['dataset'] == ds])
            status = 'OK' if n > 0 else 'EMPTY'
            print(f'   {status}  {ds:<12} : {n:,} rows')
    if 'id' not in df_meta.columns:
        print('   MISSING "id" column in CSV!')
        all_ok = False
else:
    print(f'   MISSING {META_PATH}')
    all_ok = False

# ── 3. Direct file access via CSV IDs ────────────────────────────
print('\n3. Direct file access (3 files per dataset by CSV ID):')
if df_meta is not None and 'id' in df_meta.columns:
    for ds in DATASETS:
        ds_path = DATA_PATH / ds
        if not ds_path.exists():
            print(f'   SKIP  {ds}: folder missing')
            continue

        ids = (df_meta[df_meta['dataset'] == ds]['id'].values
               if 'dataset' in df_meta.columns
               else df_meta['id'].values)

        if len(ids) == 0:
            print(f'   EMPTY  {ds}: no rows in CSV')
            continue

        # Use local disk for code15 (avoids Drive FUSE Errno 5)
        if ds == 'code15' and Path(CONFIG.get('code15_local_dir', '')).exists():
            load_path = Path(CONFIG['code15_local_dir'])
        else:
            load_path = ds_path

        ok, fail = 0, 0
        for fid in ids[:3]:
            fpath = load_path / f'{fid}.npy'
            try:
                arr = np.load(str(fpath))
                ok += 1
            except FileNotFoundError:
                print(f'   NOT FOUND  {ds}/{fid}.npy')
                fail += 1
                all_ok = False
            except Exception as e:
                print(f'   ERROR  {ds}/{fid}.npy : {e}')
                fail += 1
                all_ok = False

        if ok == 3:
            print(f'   OK   {ds:<12}: 3/3 files loaded  (CSV has {len(ids):,} IDs total)')
        else:
            print(f'   WARN {ds}: {ok}/3 files loaded')
else:
    print('   SKIPPED — CSV not loaded')

# ── 4. Checkpoint status ─────────────────────────────────────────
print('\n4. Checkpoint status:')
ckpt_file = CKPT_PATH / 'stmem_1d_checkpoint.pt'
best_file  = CKPT_PATH / 'stmem_1d_pretrained.pt'

if ckpt_file.exists():
    ckpt      = torch.load(ckpt_file, map_location='cpu', weights_only=False)
    epoch     = ckpt['epoch']
    batch     = ckpt.get('batch_idx', 0)
    complete  = ckpt.get('epoch_complete', True)
    best_loss = ckpt['best_loss']
    ts        = ckpt.get('timestamp', '?')
    print(f'   Rolling checkpoint found')
    print(f'   Epoch {epoch+1}  best_loss={best_loss:.4f}  saved={ts}')
    if complete:
        print(f'   Will START epoch {epoch+2}')
    else:
        print(f'   Will RESUME epoch {epoch+1} from batch {batch+1}')
else:
    print('   No checkpoint — will start from epoch 1')

if best_file.exists():
    best = torch.load(best_file, map_location='cpu', weights_only=False)
    print(f'   Best encoder exists: loss={best["loss"]:.4f}  epoch={best["epoch"]+1}')

# ── Verdict ───────────────────────────────────────────────────────
print('\n' + '=' * 65)
if all_ok:
    print('ALL CHECKS PASSED — safe to run Cell 7 (training)')
else:
    print('ISSUES FOUND — fix above before training')
print('=' * 65)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 4 — Deep validator  (run once before first training only)
#
# Spot-checks 20 random files per dataset for:
#   shape (12,1000)  dtype float32  no NaN  no Inf
#
# Samples random IDs from CSV and loads them by exact path.
# Does NOT list code15/.
# ══════════════════════════════════════════════════════════════════

import random
import numpy as np
import pandas as pd
from pathlib import Path

DATA_PATH    = Path(CONFIG['data_dir'])
META_PATH    = Path(CONFIG['metadata_path'])
DATASETS     = ['ptbxl', 'samitrop', 'code15']
N_SAMPLE     = 20
EXPECT_SHAPE = (12, 1000)

print('=' * 65)
print(f'DEEP VALIDATOR  ({N_SAMPLE} random files per dataset)')
print(f'Expected shape: {EXPECT_SHAPE}')
print('=' * 65)

if not META_PATH.exists():
    raise FileNotFoundError(f'CSV not found: {META_PATH}')

df_meta    = pd.read_csv(META_PATH)
all_passed = True

for ds in DATASETS:
    ds_path = DATA_PATH / ds
    print(f'\n{ds}')

    if not ds_path.exists():
        print(f'  MISSING folder: {ds_path}')
        all_passed = False
        continue

    ids = (df_meta[df_meta['dataset'] == ds]['id'].values
           if 'dataset' in df_meta.columns
           else df_meta['id'].values)

    if len(ids) == 0:
        print(f'  No CSV rows for this dataset')
        continue

    print(f'  CSV IDs : {len(ids):,}')

    sample_ids = random.sample(list(ids), min(N_SAMPLE, len(ids)))

    shapes, dtypes = [], []
    nan_count = inf_count = missing_count = err_count = 0

    # Use local disk for code15
    load_path = (Path(CONFIG.get('code15_local_dir', ''))
                 if ds == 'code15' and Path(CONFIG.get('code15_local_dir', '')).exists()
                 else ds_path)

    for fid in sample_ids:
        fpath = load_path / f'{fid}.npy'
        try:
            arr = np.load(str(fpath))
            shapes.append(arr.shape)
            dtypes.append(str(arr.dtype))
            if np.isnan(arr).any(): nan_count += 1
            if np.isinf(arr).any(): inf_count += 1
        except FileNotFoundError:
            missing_count += 1
        except Exception as e:
            print(f'  ERROR {fid}.npy: {e}')
            err_count += 1

    checked = len(sample_ids) - err_count - missing_count
    u_shapes = set(shapes)
    u_dtypes = set(dtypes)

    shape_ok   = u_shapes == {EXPECT_SHAPE}
    dtype_ok   = all(d in ('float32','float64') for d in u_dtypes)
    nan_ok     = nan_count == 0
    inf_ok     = inf_count == 0
    missing_ok = missing_count == 0

    print(f'  Checked  : {checked}/{len(sample_ids)}')
    print(f'  Shapes   : {u_shapes}  {"OK" if shape_ok else "UNEXPECTED"}')
    print(f'  Dtypes   : {u_dtypes}  {"OK" if dtype_ok else "expected float32"}')
    print(f'  NaN      : {nan_count}  {"OK" if nan_ok else "ERROR re-run normalization"}')
    print(f'  Inf      : {inf_count}  {"OK" if inf_ok else "ERROR check clipping"}')
    print(f'  Missing  : {missing_count}  {"OK" if missing_ok else "some files not on Drive"}')

    if shape_ok and dtype_ok and nan_ok and inf_ok and missing_ok:
        print(f'  PASS: {ds}')
    else:
        all_passed = False

print('\n' + '=' * 65)
print('ALL PASSED — proceed to Cell 5' if all_passed
      else 'ISSUES FOUND — fix before training')
print('=' * 65)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 5 — Model, Dataset & CheckpointManager
# ══════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import signal, time
from pathlib import Path
from datetime import datetime
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm


# ──────────────────────────────────────────────────────────────────
# Dataset — CSV-driven, never lists code15/
# ──────────────────────────────────────────────────────────────────

class STMEMSignalDataset(Dataset):

    def __init__(self, data_dir, metadata_csv,
                 code15_local_dir=None, subset=1.0, seed=42):
        self.data_dir     = Path(data_dir)
        self.signal_paths = []

        df = pd.read_csv(metadata_csv, dtype={'id': str}, low_memory=False)
        # IDs may be strings (ptbxl: '00001_hr') or ints (code15: 418484)
        # Keep as-is — f'{fid}.npy' works for both

        for ds in ['ptbxl', 'samitrop', 'code15']:
            ds_path = self.data_dir / ds

            if not ds_path.exists():
                print(f'  SKIP {ds}: folder not found at {ds_path}')
                continue

            # Use local disk for code15 to avoid Drive FUSE Errno 5
            if ds == 'code15' and code15_local_dir:
                local = Path(code15_local_dir)
                if local.exists() and any(local.iterdir()):
                    ds_path = local
                    print(f'  INFO code15: using local disk {local}')
                else:
                    print(f'  WARN code15: local dir empty/missing, using Drive')

            ids = (df[df['dataset'] == ds]['id'].values
                   if 'dataset' in df.columns
                   else df['id'].values)

            if len(ids) == 0:
                print(f'  SKIP {ds}: no IDs in CSV')
                continue

            paths = [ds_path / f'{fid}.npy' for fid in ids]
            self.signal_paths.extend(paths)
            print(f'  OK   {ds:<12}: {len(paths):,} paths  (dir: {ds_path.name})')

        if not self.signal_paths:
            raise ValueError(
                f'No signals found.\n'
                f'  data_dir     : {data_dir}\n'
                f'  metadata_csv : {metadata_csv}\n'
            )

        if subset < 1.0:
            np.random.seed(seed)
            n   = int(len(self.signal_paths) * subset)
            idx = np.random.choice(len(self.signal_paths), n, replace=False)
            self.signal_paths = [self.signal_paths[i] for i in sorted(idx)]

        print(f'\n  Total: {len(self.signal_paths):,} signals ({subset*100:.0f}%)')

    def __len__(self):
        return len(self.signal_paths)

    def __getitem__(self, idx):
        try:
            return torch.from_numpy(
                np.load(str(self.signal_paths[idx]))).float()
        except Exception:
            return torch.zeros(12, 1000)  # fallback for any missing file


# ──────────────────────────────────────────────────────────────────
# Patch Embedding
# ──────────────────────────────────────────────────────────────────

class PatchEmbed1D(nn.Module):
    def __init__(self, num_leads=12, seq_len=1000, patch_size=50, embed_dim=768):
        super().__init__()
        self.num_leads            = num_leads
        self.patch_size           = patch_size
        self.num_patches_per_lead = seq_len // patch_size   # 20
        self.num_patches          = num_leads * self.num_patches_per_lead  # 240
        self.proj       = nn.Conv1d(1, embed_dim,
                                    kernel_size=patch_size, stride=patch_size)
        self.lead_embed = nn.Parameter(torch.zeros(1, num_leads, 1, embed_dim))
        nn.init.trunc_normal_(self.lead_embed, std=0.02)

    def forward(self, x):
        B, L, T = x.shape                                         # (B,12,1000)
        x = self.proj(x.view(B * L, 1, T))                       # (B*12,D,20)
        x = x.view(B, L, -1, self.num_patches_per_lead)          # (B,12,D,20)
        x = x.permute(0, 1, 3, 2)                                # (B,12,20,D)
        x = (x + self.lead_embed).reshape(B, self.num_patches, -1)  # (B,240,D)
        return x


# ──────────────────────────────────────────────────────────────────
# ST-MEM 1D
#
# Sequence  (252 tokens):
#   [CLS]  L0×20  [SEP]  L1×20  [SEP]  ...  L10×20  [SEP]  L11×20
#     1     20      1     20      1           20       1      20
#   = 1 + 11*(20+1) + 20 = 252
#
# pos_embed  : (1, 252, 768)
# sep_tokens : (1,  11, 768)
# Masking    : 75% of the 240 patch tokens only
# ──────────────────────────────────────────────────────────────────

class STMEM1D(nn.Module):

    def __init__(self, embed_dim=768, depth=12, num_heads=12,
                 decoder_embed_dim=512, decoder_depth=4,
                 decoder_num_heads=8, mask_ratio=0.75):
        super().__init__()

        self.patch_embed      = PatchEmbed1D(embed_dim=embed_dim)
        self.num_leads        = self.patch_embed.num_leads             # 12
        self.patches_per_lead = self.patch_embed.num_patches_per_lead  # 20
        self.num_patches      = self.patch_embed.num_patches           # 240
        self.num_seps         = self.num_leads - 1                     # 11
        self.mask_ratio       = mask_ratio

        # 1 + 240 + 11 = 252
        seq_len = 1 + self.num_patches + self.num_seps

        self.cls_token  = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.sep_tokens = nn.Parameter(torch.zeros(1, self.num_seps, embed_dim))
        self.pos_embed  = nn.Parameter(torch.zeros(1, seq_len, embed_dim))

        self.encoder = nn.ModuleList([
            nn.TransformerEncoderLayer(
                embed_dim, num_heads,
                dim_feedforward=embed_dim * 4,
                dropout=0.0, activation='gelu',
                batch_first=True, norm_first=True)
            for _ in range(depth)
        ])
        self.encoder_norm = nn.LayerNorm(embed_dim)

        self.decoder_embed     = nn.Linear(embed_dim, decoder_embed_dim)
        self.mask_token        = nn.Parameter(
            torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_pos_embed = nn.Parameter(
            torch.zeros(1, seq_len, decoder_embed_dim))
        self.decoder = nn.ModuleList([
            nn.TransformerEncoderLayer(
                decoder_embed_dim, decoder_num_heads,
                dim_feedforward=decoder_embed_dim * 4,
                dropout=0.0, activation='gelu',
                batch_first=True, norm_first=True)
            for _ in range(decoder_depth)
        ])
        self.decoder_norm = nn.LayerNorm(decoder_embed_dim)
        self.decoder_pred = nn.Linear(decoder_embed_dim,
                                       self.patch_embed.patch_size)

        for p in [self.cls_token, self.sep_tokens, self.pos_embed,
                  self.mask_token, self.decoder_pos_embed]:
            nn.init.trunc_normal_(p, std=0.02)

    def _build_sequence(self, patches, B):
        """[CLS] L0 [SEP] L1 [SEP] ... L10 [SEP] L11  ->  (B,252,D)"""
        D     = patches.shape[-1]
        leads = patches.reshape(B, self.num_leads, self.patches_per_lead, D)
        parts = [self.cls_token.expand(B, -1, -1)]
        for i in range(self.num_leads):
            parts.append(leads[:, i])
            if i < self.num_leads - 1:
                parts.append(self.sep_tokens[:, i:i+1].expand(B, -1, -1))
        return torch.cat(parts, dim=1)

    def _patch_positions(self):
        """Returns the 240 token indices that are patch tokens (not CLS/SEP)."""
        positions, pos = [], 1
        for i in range(self.num_leads):
            for _ in range(self.patches_per_lead):
                positions.append(pos)
                pos += 1
            if i < self.num_leads - 1:
                pos += 1   # skip SEP token
        return positions   # exactly 240 values

    def forward(self, signals):
        B = signals.shape[0]
        D = self.pos_embed.shape[-1]

        patches = self.patch_embed(signals)         # (B,240,D)
        x       = self._build_sequence(patches, B)  # (B,252,D)
        x       = x + self.pos_embed

        patch_pos  = self._patch_positions()         # 240 indices
        N          = len(patch_pos)
        num_masked = int(N * self.mask_ratio)
        num_vis    = N - num_masked

        noise       = torch.rand(B, N, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        ids_keep    = ids_shuffle[:, :num_vis]

        patch_pos_t   = torch.tensor(patch_pos, device=x.device)
        patch_tokens  = x[:, patch_pos_t]

        non_patch_pos = [p for p in range(x.shape[1])
                         if p not in set(patch_pos)]
        non_patch_t   = torch.tensor(non_patch_pos, device=x.device)

        vis = torch.gather(
            patch_tokens, 1,
            ids_keep.unsqueeze(-1).expand(-1, -1, D))
        x_vis = torch.cat([x[:, non_patch_t], vis], dim=1)

        for layer in self.encoder: x_vis = layer(x_vis)
        x_vis = self.encoder_norm(x_vis)

        x_dec    = self.decoder_embed(x_vis)
        n_non    = len(non_patch_pos)
        enc_vis  = x_dec[:, n_non:]
        dec_dim  = self.mask_token.shape[-1]

        full = self.mask_token.expand(B, N, -1).clone()
        full.scatter_(
            1, ids_keep.unsqueeze(-1).expand(-1, -1, dec_dim),
            enc_vis.to(full.dtype))  # cast: AMP makes enc_vis fp16, full is fp32
        full = torch.gather(
            full, 1, ids_restore.unsqueeze(-1).expand(-1, -1, dec_dim))
        full = full + self.decoder_pos_embed[:, patch_pos_t, :]

        for layer in self.decoder: full = layer(full)
        pred = self.decoder_pred(self.decoder_norm(full))  # (B,240,50)

        target = signals.reshape(
            B, self.num_leads,
            self.patches_per_lead,
            self.patch_embed.patch_size
        ).reshape(B, N, self.patch_embed.patch_size)

        mask = torch.zeros(B, N, device=signals.device)
        mask.scatter_(1, ids_shuffle[:, :num_masked], 1)

        loss = ((pred - target) ** 2 * mask.unsqueeze(-1)).sum()
        loss = loss / mask.sum() / self.patch_embed.patch_size

        return loss, pred, mask


# ──────────────────────────────────────────────────────────────────
# Checkpoint Manager
# ──────────────────────────────────────────────────────────────────

class CheckpointManager:

    def __init__(self, checkpoint_dir):
        self.ckpt_path   = Path(checkpoint_dir) / 'stmem_1d_checkpoint.pt'
        self.best_path   = Path(checkpoint_dir) / 'stmem_1d_pretrained.pt'
        self.loss_log    = []
        self.interrupted = False
        Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)

        def _handler(sig, frame):
            print('\nInterrupt — saving then stopping...')
            self.interrupted = True
        signal.signal(signal.SIGINT, _handler)

    def save(self, epoch, batch_idx, total_batches,
             model, optimizer, scheduler, scaler,
             loss, best_loss, is_best=False, epoch_complete=False):

        ckpt = {
            'epoch'               : epoch,
            'batch_idx'           : batch_idx,
            'total_batches'       : total_batches,
            'epoch_complete'      : epoch_complete,
            'model_state_dict'    : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict'   : scaler.state_dict() if scaler else None,
            'loss'                : loss,
            'best_loss'           : best_loss,
            'loss_log'            : self.loss_log,
            'timestamp'           : datetime.now().isoformat(),
        }
        tmp = self.ckpt_path.with_suffix('.tmp')
        torch.save(ckpt, tmp)
        tmp.replace(self.ckpt_path)   # atomic

        status = ('complete' if epoch_complete
                  else f'batch {batch_idx}/{total_batches}')
        print(f'  Saved  epoch {epoch+1} | {status} | loss {loss:.4f}')

        if is_best:
            torch.save({
                'epoch': epoch,
                'loss' : loss,
                'model_state_dict': {
                    'patch_embed'  : model.patch_embed.state_dict(),
                    'cls_token'    : model.cls_token.data,
                    'pos_embed'    : model.pos_embed.data,
                    'sep_tokens'   : model.sep_tokens.data,
                    'encoder'      : model.encoder.state_dict(),
                    'encoder_norm' : model.encoder_norm.state_dict(),
                }
            }, self.best_path)
            print(f'  Best   loss {loss:.4f} -> stmem_1d_pretrained.pt')

    def load(self, model, optimizer, scheduler, scaler, device):
        if not self.ckpt_path.exists():
            print('  No checkpoint — starting from epoch 1')
            return 0, 0, float('inf')

        print(f'  Loading: {self.ckpt_path}')
        ckpt = torch.load(self.ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        if scaler and ckpt.get('scaler_state_dict'):
            scaler.load_state_dict(ckpt['scaler_state_dict'])

        self.loss_log = ckpt.get('loss_log', [])
        epoch    = ckpt['epoch']
        batch    = ckpt.get('batch_idx', 0)
        best     = ckpt['best_loss']
        complete = ckpt.get('epoch_complete', True)

        print(f'  Best loss : {best:.4f}')
        print(f'  Saved at  : {ckpt.get("timestamp", "?")}')
        if complete:
            print(f'  Epoch {epoch+1} complete -> starting epoch {epoch+2}')
            return epoch + 1, 0, best
        else:
            print(f'  Resuming epoch {epoch+1} from batch {batch+1}')
            return epoch, batch + 1, best


# ── Sanity check ─────────────────────────────────────────────────
print('Checking architecture...')
_m = STMEM1D()
_x = torch.randn(2, 12, 1000)
_l, _, _ = _m(_x)
print(f'  pos_embed  : {tuple(_m.pos_embed.shape)}   expected (1, 252, 768)')
print(f'  sep_tokens : {tuple(_m.sep_tokens.shape)}   expected (1, 11, 768)')
print(f'  Forward OK : loss = {_l.item():.4f}')
assert _m.pos_embed.shape[1] == 252, \
    f'pos_embed wrong: got {_m.pos_embed.shape[1]}, expected 252'
del _m, _x, _l
print('All classes defined and verified OK')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 6 — Training function + keepalive
# ══════════════════════════════════════════════════════════════════

import threading, numpy as np, time, torch
from pathlib import Path
from datetime import datetime
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


def colab_keepalive(interval_sec=1800):
    """Heartbeat every 30 min to prevent Colab idle disconnect."""
    def _beat():
        count = 0
        while True:
            time.sleep(interval_sec)
            count += 1
            print(f'  Keepalive #{count} {datetime.now().strftime("%H:%M:%S")}')
    threading.Thread(target=_beat, daemon=True).start()
    print('Keepalive started (every 30 min)')


colab_keepalive()


def train_stmem(cfg):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f'\n{"="*65}')
    print('ST-MEM 1D Pretraining')
    print(f'{"="*65}')
    if device.type == 'cuda':
        print(f'GPU   : {torch.cuda.get_device_name(0)}')
        print(f'VRAM  : '
              f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    print(f'AMP   : {cfg["use_amp"]}  '
          f'Epochs: {cfg["epochs"]}  '
          f'Batch: {cfg["batch_size"]}  '
          f'LR: {cfg["lr"]}')
    print(f'{"="*65}\n')

    # ── Dataset ───────────────────────────────────────────────────
    print('Building dataset from CSV...')
    dataset = STMEMSignalDataset(
        cfg['data_dir'],
        metadata_csv=cfg['metadata_path'],
        code15_local_dir=cfg.get('code15_local_dir'),
        subset=cfg['subset'],
    )
    loader = DataLoader(
        dataset,
        batch_size=cfg['batch_size'],
        shuffle=True,
        num_workers=cfg['num_workers'],
        pin_memory=(device.type == 'cuda'),
        drop_last=True,
        persistent_workers=(cfg['num_workers'] > 0),
        prefetch_factor=2 if cfg['num_workers'] > 0 else None,
    )
    total_batches = len(loader)
    print(f'  {total_batches:,} batches per epoch\n')

    # ── Model ─────────────────────────────────────────────────────
    model = STMEM1D(mask_ratio=cfg['mask_ratio']).to(device)
    total = sum(p.numel() for p in model.parameters())
    enc   = sum(p.numel() for n, p in model.named_parameters()
                if not any(x in n for x in ['decoder', 'mask_token']))
    print(f'[MODEL] Total={total:,}  Encoder={enc:,}  Decoder={total-enc:,}\n')

    # ── Optimizer ─────────────────────────────────────────────────
    try:
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=cfg['lr'],
            betas=(0.9, 0.95), weight_decay=0.05, fused=True)
        print('Fused AdamW')
    except TypeError:
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=cfg['lr'],
            betas=(0.9, 0.95), weight_decay=0.05)
        print('Standard AdamW')

    def lr_lambda(ep):
        if ep < cfg['warmup_epochs']:
            return (ep + 1) / cfg['warmup_epochs']
        p = ((ep - cfg['warmup_epochs'])
             / max(cfg['epochs'] - cfg['warmup_epochs'], 1))
        return 0.5 * (1 + np.cos(np.pi * p))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    scaler = None
    if cfg['use_amp'] and device.type == 'cuda':
        try:
            scaler = torch.amp.GradScaler('cuda')
        except TypeError:
            scaler = torch.cuda.amp.GradScaler()

    # ── Load checkpoint ───────────────────────────────────────────
    ckpt_mgr = CheckpointManager(cfg['checkpoint_dir'])
    start_epoch, start_batch, best_loss = ckpt_mgr.load(
        model, optimizer, scheduler, scaler, device)

    if start_epoch >= cfg['epochs']:
        print(f'Training already complete. Best loss: {best_loss:.4f}')
        return

    print(f'\nStarting epoch {start_epoch+1}/{cfg["epochs"]}  '
          f'batch {start_batch}/{total_batches}\n')

    t0 = time.time()

    for epoch in range(start_epoch, cfg['epochs']):
        model.train()
        epoch_loss, n_done = 0.0, 0
        t_ep = time.time()

        pbar = tqdm(loader, desc=f'Epoch {epoch+1}/{cfg["epochs"]}')

        for batch_idx, signals in enumerate(pbar):

            if epoch == start_epoch and batch_idx < start_batch:
                if batch_idx % 2000 == 0:
                    pbar.set_postfix({'skip': f'->{start_batch}'})
                continue

            signals = signals.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            if scaler:
                try:
                    ctx = torch.amp.autocast('cuda')
                except Exception:
                    ctx = torch.cuda.amp.autocast()
                with ctx:
                    loss, _, _ = model(signals)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss, _, _ = model(signals)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            epoch_loss += loss.item()
            n_done     += 1
            avg         = epoch_loss / n_done

            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'avg' : f'{avg:.4f}',
                'best': f'{best_loss:.4f}',
                'lr'  : f'{optimizer.param_groups[0]["lr"]:.2e}',
            })

            if (batch_idx + 1) % cfg['checkpoint_every_batches'] == 0:
                is_best = avg < best_loss
                if is_best: best_loss = avg
                ckpt_mgr.loss_log.append(
                    {'epoch': epoch, 'batch': batch_idx, 'loss': avg})
                ckpt_mgr.save(epoch, batch_idx, total_batches,
                              model, optimizer, scheduler, scaler,
                              avg, best_loss,
                              is_best=is_best, epoch_complete=False)

            if ckpt_mgr.interrupted:
                avg = epoch_loss / max(n_done, 1)
                ckpt_mgr.loss_log.append(
                    {'epoch': epoch, 'batch': batch_idx, 'loss': avg})
                ckpt_mgr.save(epoch, batch_idx, total_batches,
                              model, optimizer, scheduler, scaler,
                              avg, best_loss,
                              is_best=False, epoch_complete=False)
                print('\nStopped. Re-run Cell 7 to resume.')
                return

        # ── End of epoch ──────────────────────────────────────────
        start_batch = 0
        scheduler.step()

        avg     = epoch_loss / max(n_done, 1)
        is_best = avg < best_loss
        if is_best: best_loss = avg

        elapsed = time.time() - t0
        done    = epoch - start_epoch + 1
        eta_h   = (elapsed / done) * (cfg['epochs'] - epoch - 1) / 3600

        ckpt_mgr.loss_log.append({'epoch': epoch, 'batch': 'end', 'loss': avg})
        ckpt_mgr.save(epoch, total_batches - 1, total_batches,
                      model, optimizer, scheduler, scaler,
                      avg, best_loss, is_best=is_best, epoch_complete=True)

        print(f'[EP {epoch+1:02d}/{cfg["epochs"]}]  '
              f'loss={avg:.4f}  '
              f'lr={optimizer.param_groups[0]["lr"]:.2e}  '
              f'time={(time.time()-t_ep)/60:.1f}min  '
              f'ETA={eta_h:.1f}h')

        if (epoch + 1) % cfg['save_every'] == 0:
            ms = (Path(cfg['checkpoint_dir'])
                  / f'stmem_1d_epoch{epoch+1:02d}.pt')
            torch.save({
                'epoch': epoch, 'loss': avg,
                'model_state_dict': {
                    'patch_embed'  : model.patch_embed.state_dict(),
                    'cls_token'    : model.cls_token.data,
                    'pos_embed'    : model.pos_embed.data,
                    'sep_tokens'   : model.sep_tokens.data,
                    'encoder'      : model.encoder.state_dict(),
                    'encoder_norm' : model.encoder_norm.state_dict(),
                }
            }, ms)
            print(f'  Milestone: {ms.name}')

    total_h = (time.time() - t0) / 3600
    print(f'\nDone! {total_h:.1f}h  Best loss: {best_loss:.4f}')
    print(f'Encoder saved -> stmem_1d_pretrained.pt')


print('Training function ready')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 7 — START / RESUME TRAINING
#
# After disconnect: re-run Cells 1 -> 2 -> 5 -> 6 -> this cell.
# Training resumes automatically from the last checkpoint.
# ══════════════════════════════════════════════════════════════════

train_stmem(CONFIG)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 8 — Verify output after training
# ══════════════════════════════════════════════════════════════════

import torch
from pathlib import Path

CKPT_DIR = Path(CONFIG['checkpoint_dir'])

print('=' * 65)
print('CHECKPOINT STATUS')
print('=' * 65)

for fname, label in [
    ('stmem_1d_checkpoint.pt', 'Rolling checkpoint'),
    ('stmem_1d_pretrained.pt', 'Best encoder (use for fine-tuning)'),
]:
    f = CKPT_DIR / fname
    if f.exists():
        c        = torch.load(f, map_location='cpu', weights_only=False)
        complete = c.get('epoch_complete', True)
        print(f'\nOK  {label}')
        print(f'  Epoch  : {c["epoch"]+1}')
        print(f'  Loss   : {c["loss"]:.4f}')
        if 'best_loss' in c:
            print(f'  Best   : {c["best_loss"]:.4f}')
        if fname == 'stmem_1d_pretrained.pt':
            sd  = c['model_state_dict']
            pos = sd['pos_embed'].shape
            ok  = 'OK' if pos[1] == 252 else f'WRONG expected (1,252,768)'
            print(f'  Keys   : {list(sd.keys())}')
            print(f'  pos_embed: {tuple(pos)}  {ok}')
            print(f'  Size   : {f.stat().st_size/1e6:.1f} MB')
        status = 'complete' if complete else f'interrupted @ batch {c.get("batch_idx")}'
        print(f'  Status : {status}')
    else:
        print(f'\nMISSING  {label}')

milestones = sorted(CKPT_DIR.glob('stmem_1d_epoch*.pt'))
if milestones:
    print(f'\nMilestones ({len(milestones)}):')
    for m in milestones:
        c = torch.load(m, map_location='cpu', weights_only=False)
        print(f'  {m.name:<40} loss={c["loss"]:.4f}')

print('\n' + '=' * 65)
print('Next steps:')
print('  1. Download stmem_1d_pretrained.pt from Drive')
print('  2. Copy to local: checkpoints/stmem_1d_pretrained.pt')
print('  3. load_stmem_pretrained("checkpoints/stmem_1d_pretrained.pt")')
print('=' * 65)